# Day 2 — Attention visualizer

Pick GPT-2 small (12 layers × 12 heads — small enough to see, big enough to be interesting) and plot attention patterns. Identify 2-3 heads with visually distinct behavior.

We use [`transformer_lens`](https://github.com/TransformerLensOrg/TransformerLens) because `model.run_with_cache(prompt)` gives you every activation in one call. If install fails, fall back to plain HF with `output_attentions=True`.

In [ ]:
# !pip install transformer_lens matplotlib seaborn

import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2")
model.eval()
print("layers:", model.cfg.n_layers, "heads/layer:", model.cfg.n_heads)

In [ ]:
prompt = "The Eiffel Tower is in Paris. The Statue of Liberty is in"
tokens = model.to_tokens(prompt)
str_tokens = model.to_str_tokens(prompt)
print(str_tokens)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

# next-token prediction sanity check
next_id = logits[0, -1].argmax().item()
print("next token:", repr(model.tokenizer.decode([next_id])))

## Plot attention heatmaps for layers × heads

In [ ]:
def plot_attention(layer: int, heads=(0, 5, 11)):
    attn = cache["pattern", layer]  # [batch, head, q, k]
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 4))
    for ax, h in zip(axes, heads):
        sns.heatmap(
            attn[0, h].cpu().numpy(),
            xticklabels=str_tokens, yticklabels=str_tokens,
            ax=ax, cbar=False, cmap="viridis", square=True,
        )
        ax.set_title(f"L{layer}H{h}")
        ax.set_xlabel("key"); ax.set_ylabel("query")
    plt.tight_layout()
    plt.show()

plot_attention(layer=0)
plot_attention(layer=5)

## Observations to write

After running the cells above:
1. Find a head that attends primarily to the **previous token** (a 'shift-by-1' head). Note its (layer, head) index.
2. Find a head that attends primarily to the **first token / BOS**. Many heads do this — it's an attention-sink phenomenon.
3. Try the prompt `"A B C A B"` repeated as in the [induction-head literature](https://transformer-circuits.pub/2021/framework/index.html). Identify a head with diagonal-offset attention — that's an induction head.
4. Write 3 bullets summarizing what you saw. This becomes Day 2's section of your blog draft.